In [1]:
!pip install -q "trl>=0.15.0" "transformers>=5.0.0" accelerate datasets peft qwen-vl-utils

In [2]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
%cd /content/drive/MyDrive/RL/RL_Scene_Graphs
%ls

/content/drive/MyDrive/RL/RL_Scene_Graphs
datasets/  Image_Data_Prep.ipynb  mllm_generation.py     __pycache__/
grpo-run/  MLLM_Eval.ipynb        MLLM_Pipeline.ipynb
grpo-sgg/  mllm_eval.py           MLLM_Train_GRPO.ipynb


In [4]:
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/RL/RL_Scene_Graphs")  # change this
sys.path.append(str(PROJECT_DIR))

In [5]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

### Dataset and Prompt

In [6]:
import torch
import torch.nn.functional as F
from datasets import load_from_disk, Dataset
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, TrainingArguments, Trainer
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType
from tqdm import tqdm
import json

from mllm_generation import (
    build_messages,
    build_inputs,
    extract_answer_content
)
from mllm_eval import total_reward
import ast

### Load Dataset

In [7]:
# iterable_ds = load_from_disk("./datasets/vg150_val_sgg_prompt")
iterable_ds = load_from_disk("./datasets/vg150_val_sgg_prompt").to_iterable_dataset()

train_ds = Dataset.from_list(list(iterable_ds.take(1500)))
# test_ds = ds.select(range(3500, len(ds)))

# def to_grpo_row(sample):
#     return {
#         "image": sample["image"].convert("RGB"),
#         "prompt": sample.get("prompt_close", ""),
#         "gt_objects": sample["objects"],
#         "gt_relationships": sample["relationships"],
#     }

# ds = ds.map(to_grpo_row, remove_columns=ds.column_names)
print(train_ds)
# print(test_ds)

print("Training examples: ", len(train_ds))
print(train_ds[0])
# print("Test examples: ", len(test_ds))
# print(test_ds[0])
# print an example


Dataset({
    features: ['image_id', 'image', 'prompt_open', 'prompt_close', 'objects', 'relationships'],
    num_rows: 1500
})
Training examples:  1500
{'image_id': '1', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=800x600 at 0x7E1F3A3504D0>, 'prompt_open': 'Generate a structured scene graph for an image of size (800 x 600) using the following format:\n\n<answer>\n{\n  "objects": [\n    {"id": "object_name.number", "bbox": [x1, y1, x2, y2]},\n    ...\n  ],\n  "relationships": [\n    {"subject": "object_name.number", "predicate": "relationship_type", "object": "object_name.number"},\n    ...\n  ]\n}\n</answer>\n\n### **Guidelines:**\n- **Objects:**\n  - Assign a unique ID for each object using the format `"object_name.number"` (e.g., `"person.1"`, `"bike.2"`).\n  - Provide its bounding box `[x1, y1, x2, y2]` in integer pixel format.\n  - Include all visible objects, even if they have no relationships.\n\n- **Relationships:**\n  - Represent interactions accurately usi

In [8]:
sample_orig = train_ds[0]
print(sample_orig)

{'image_id': '1', 'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=800x600 at 0x7E1F3A350BC0>, 'prompt_open': 'Generate a structured scene graph for an image of size (800 x 600) using the following format:\n\n<answer>\n{\n  "objects": [\n    {"id": "object_name.number", "bbox": [x1, y1, x2, y2]},\n    ...\n  ],\n  "relationships": [\n    {"subject": "object_name.number", "predicate": "relationship_type", "object": "object_name.number"},\n    ...\n  ]\n}\n</answer>\n\n### **Guidelines:**\n- **Objects:**\n  - Assign a unique ID for each object using the format `"object_name.number"` (e.g., `"person.1"`, `"bike.2"`).\n  - Provide its bounding box `[x1, y1, x2, y2]` in integer pixel format.\n  - Include all visible objects, even if they have no relationships.\n\n- **Relationships:**\n  - Represent interactions accurately using `"subject"`, `"predicate"`, and `"object"`.\n  - Omit relationships for orphan objects.\n\n### **Example Output:**\n<answer>\n{\n  "objects": [\n    {

In [10]:
PROMPT_TEMPLATE = """Generate a scene graph for the image of size ({width} x {height}).

Output ONLY:

<answer>
{{
  "objects": [{{"id": "name.number", "bbox": [x1, y1, x2, y2]}}],
  "relationships": [{{"subject": "name.number", "predicate": "relation", "object": "name.number"}}]
}}
</answer>

Rules:
- Unique ids: name.number
- Bounding boxes must be integers
- Include all visible objects
- Relationships must reference valid object ids
- Output valid JSON only
"""

def build_dataset(raw_ds):
    data = []

    for sample in tqdm(raw_ds, total=len(raw_ds)):
        width, height = sample["image"].size
        prompt = PROMPT_TEMPLATE.format(width=width, height=height)
        data.append({
            "prompt": sample["prompt_close"],
            # "prompt": prompt,
            "image_data": sample["image"],   # rename
            "solution": {
                "objects": ast.literal_eval(sample["objects"]),
                "relationships": ast.literal_eval(sample["relationships"])
            }
        })

    return Dataset.from_list(data)

train_ds = build_dataset(train_ds)
# test_ds = build_dataset(test_ds)

print(train_ds)
# print(test_ds)



100%|██████████| 1500/1500 [00:05<00:00, 256.04it/s]


Dataset({
    features: ['prompt', 'image_data', 'solution'],
    num_rows: 1500
})


In [11]:
# one example each
print(train_ds[0])
# print(test_ds[0])
print(type(train_ds[0]["solution"]["objects"]))
print(type(train_ds[0]["solution"]["relationships"]))

{'prompt': 'Generate a structured scene graph for an image of size (800 x 600) using the specified object and relationship categories.\n\n### **Output Format:**\n<answer>\n{\n  "objects": [\n    {"id": "object_name.number", "bbox": [x1, y1, x2, y2]},\n    ...\n  ],\n  "relationships": [\n    {"subject": "object_name.number", "predicate": "relationship_type", "object": "object_name.number"},\n    ...\n  ]\n}\n</answer>\n\n### **Guidelines:**\n- **Objects:**\n  - Assign unique IDs in the format `"object_name.number"` (e.g., `"person.1"`). The **object_name** must belong to the predefined object set: `["airplane", "animal", "arm", "bag", "banana", "basket", "beach", "bear", "bed", "bench", "bike", "bird", "board", "boat", "book", "boot", "bottle", "bowl", "box", "boy", "branch", "building", "bus", "cabinet", "cap", "car", "cat", "chair", "child", "clock", "coat", "counter", "cow", "cup", "curtain", "desk", "dog", "door", "drawer", "ear", "elephant", "engine", "eye", "face", "fence", "fing

In [12]:
SYSTEM_PROMPT = (
    "A conversation between User and Assistant. The user asks a question, and the Assistant solves it. The assistant "
    "first thinks about the reasoning process in the mind and then provides the user with the answer. The reasoning "
    "process and answer are enclosed within <think> </think> and <answer> </answer> tags, respectively, i.e., "
    "<think> reasoning process here </think><answer> answer here </answer>"
)

### Reward

In [13]:
def grpo_reward(completions, solution, **kwargs):
    rewards = []

    for i, (completion, gt) in enumerate(zip(completions, solution)):
        if isinstance(completion, str):
            text = completion
        else:
            text = completion[0]["content"]

        try:
            parsed = extract_answer_content(text)
            pred = json.loads(parsed)
        except:
            pred = {"_raw_text": text}

        print(f"\n--- SAMPLE {i} ---")
        print("TEXT:", text)
        print("Reward with GT: ", gt, type(gt), " and pred ", pred, type(pred))

        r = total_reward(pred, gt)['total']


        print("REWARD:", r)

        rewards.append(r)

    return rewards

### Custom Trainer

In [14]:
class GRPOConfigVL(TrainingArguments):
    def __init__(
        self,
        num_generations=2,
        max_completion_length=512,
        beta=0.01,
        temperature=0.8,
        top_p=0.9,
        **kwargs
    ):
        super().__init__(**kwargs)
        self.num_generations = num_generations
        self.max_completion_length = max_completion_length
        self.beta = beta
        self.temperature = temperature
        self.top_p = top_p

In [15]:
class GRPOTrainerVL(Trainer):

  def __init__(
        self,
        model,
        args,
        train_dataset=None,
        eval_dataset=None,
        processing_class=None,
        reward_funcs=None,
        **kwargs,
    ):
        super().__init__(
            model=model,
            args=args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            **kwargs,
        )

        # 🔥 custom components
        self.processing_class = processing_class
        self.reward_funcs = reward_funcs

  def generate_completions(self, batch):
    prompts = batch["prompt"]
    images = batch["image_data"]

    all_prompt_ids = []
    all_completion_ids = []
    all_texts = []

    for prompt, image in zip(prompts, images):

        for ind in range(self.args.num_generations):

            print("GEN ", ind)

            messages = build_messages(image, prompt, system_prompt=SYSTEM_PROMPT)
            inputs, _ = build_inputs(self.processing_class, messages, self.model.device)

            self.model.eval()
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=self.args.max_completion_length,
                    do_sample=True,
                    temperature=self.args.temperature,
                    top_p=self.args.top_p,
                )
            self.model.train()

            input_ids = inputs["input_ids"][0]
            gen_ids = outputs[0, input_ids.shape[0]:]

            text = self.processing_class.batch_decode(
                [gen_ids], skip_special_tokens=True
            )[0]

            all_prompt_ids.append(input_ids)
            all_completion_ids.append(gen_ids)
            all_texts.append(text)

    print("GENERATED --- ")
    print(len(all_prompt_ids), " = prompt ids ", type(all_prompt_ids[0]))
    print(len(all_completion_ids), " = completion ids ", type(all_completion_ids[0]))
    print(len(all_texts), " = texts")
    return all_prompt_ids, all_completion_ids, all_texts


  def compute_logprobs(self, prompt_ids, completion_ids):
    device = self.model.device

    # ===== Build full sequence =====
    full_ids = torch.cat([prompt_ids, completion_ids], dim=0).unsqueeze(0)
    attention_mask = torch.ones_like(full_ids)

    prompt_len = prompt_ids.shape[0]
    completion_len = completion_ids.shape[0]

    # ===== Build labels mask =====
    labels = full_ids.clone()

    # mask prompt tokens → ignored in loss
    labels[:, :prompt_len] = -100

    # ===== Forward pass (WITH grad) =====
    outputs = self.model(
        input_ids=full_ids,
        attention_mask=attention_mask,
        labels=labels,   # 🔥 KEY
    )

    # logits: (1, P+C, V)
    logits = outputs.logits
    log_probs = torch.log_softmax(logits, dim=-1)

    # ===== Extract ONLY completion token logprobs =====
    token_logprobs = []

    for t in range(completion_len):
        token_id = int(completion_ids[t])
        pos = prompt_len + t - 1
        lp = log_probs[0, pos, token_id]
        token_logprobs.append(lp)

    return torch.stack(token_logprobs)  # (C,)

  def compute_rewards(self, prompts, completions, batch):

    expanded_solutions = []

    for sol in batch["solution"]:      # length B
        for _ in range(self.args.num_generations):
            expanded_solutions.append(sol)

    rewards = self.reward_funcs(
        completions=[[{"content": c}] for c in completions],
        solution=expanded_solutions
    )
    return torch.tensor(rewards, device=self.model.device)

  def compute_loss(self, model, inputs, return_outputs=False, **kwargs):

    prompts = inputs["prompt"]
    images = inputs["image_data"]

    prompt_ids_list, completion_ids_list, texts = self.generate_completions(inputs)

    # ===== rewards =====
    rewards = self.compute_rewards(prompts, texts, inputs)

    # ===== reshape into groups =====
    G = self.args.num_generations
    rewards = rewards.view(-1, G)

    advantages = rewards - rewards.mean(dim=1, keepdim=True)
    advantages = advantages.view(-1)

    # ===== compute logprobs =====
    all_logprobs = []

    for p_ids, c_ids in zip(prompt_ids_list, completion_ids_list):
        lp = self.compute_logprobs(p_ids, c_ids)
        all_logprobs.append(lp.sum())  # sequence logprob

    logprobs = torch.stack(all_logprobs)

    print("Rewards:", rewards[:5])
    print("Advantages:", advantages[:5])
    print("Logprobs:", logprobs[:5])

    # ===== GRPO loss =====
    loss = -(advantages * logprobs).mean()
    print("loss requires grad:", loss.requires_grad)

    return loss

In [ ]:
# def qwen_vl_rollout(prompts, trainer):
#     """
#     prompts: list[str]
#     trainer: GRPOTrainer instance
#     """

#     model = trainer.model
#     processor = trainer.processing_class
#     device = model.device

#     dataset_batch = trainer._current_inputs  # we will inject this

#     all_prompt_ids = []
#     all_completion_ids = []
#     all_logprobs = []
#     all_texts = []

#     for i, prompt in enumerate(prompts):
#         image = dataset_batch[i]["image_data"]

#         for _ in range(trainer.num_generations):

#             # 🔥 YOUR PIPELINE
#             messages = build_messages(image, prompt, system_prompt=SYSTEM_PROMPT)

#             inputs, input_text = build_inputs(processor, messages, device)

#             model.eval()
#             with torch.no_grad():
#                 generated_ids = model.generate(
#                     **inputs,
#                     max_new_tokens=trainer.args.max_completion_length,
#                     # do_sample=True,
#                     # temperature=trainer.args.temperature,
#                     # top_p=trainer.args.top_p,
#                 )

#             model.train()

#             # trim prompt
#             input_ids = inputs["input_ids"]

#             print("\n=== RAW GENERATED IDS ===")
#             print(generated_ids)

#             print("\n=== INPUT IDS LENGTH ===")
#             print([len(x) for x in input_ids])

#             print("\n=== OUTPUT IDS LENGTH ===")
#             print([len(x) for x in generated_ids])

#             # print(input_ids)
#             generated_ids_trimmed = [out_ids[len(in_ids):] for in_ids, out_ids in zip(input_ids, generated_ids)]
#             text = processor.batch_decode(generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False)[0]

#             gen_ids = generated_ids_trimmed[0] # since batch size is 1
#             prompt_ids = input_ids[0]


#             # get log probs
#             full_ids = torch.cat([prompt_ids, gen_ids], dim=0).unsqueeze(0).to(device)
#             attention_mask = torch.ones_like(full_ids).to(device)

#             outputs = model(
#                 input_ids=full_ids,
#                 attention_mask=attention_mask,
#             )

#             logits = outputs.logits
#             log_probs = torch.log_softmax(logits, dim=-1)

#             completion_logprobs = []
#             prompt_len = prompt_ids.shape[0]

#             for t in range(len(gen_ids)):
#                 token_id = int(gen_ids[t].item())
#                 pos = prompt_len + t - 1
#                 lp = log_probs[0, pos, token_id]
#                 completion_logprobs.append(float(lp.item()))


#             print("Appending ", len(completion_logprobs), " log probs")
#             all_logprobs.append(completion_logprobs)

#             # store
#             print("Appending ", len(prompt_ids), " prompt ids")
#             all_prompt_ids.append(prompt_ids.tolist())

#             print("Appending ", len(gen_ids.tolist()), " completion ids")
#             all_completion_ids.append(gen_ids.tolist())
#             # all_logprobs.append([0.0] * len(gen_ids))  # placeholder
#             all_texts.append(text)

#             print("\n=== GENERATION DEBUG ===", flush=True)
#             # print("INPUT PROMPT:", input_text)
#             print("OUTPUT:", text)


#     assert len(all_prompt_ids) == len(all_completion_ids) == len(all_logprobs)
#     print(type(all_completion_ids))
#     print(type(all_completion_ids[0]))
#     print(type(all_completion_ids[0][0]))

#     print(type(all_logprobs[0][0]))

#     print(type(all_prompt_ids))
#     print(type(all_prompt_ids[0]))
#     print(type(all_prompt_ids[0][0]))

#     return {
#         "prompt_ids": all_prompt_ids,
#         "completion_ids": all_completion_ids,
#         "logprobs": all_logprobs,
#         # "completions": [[{"content": t}] for t in all_texts],
#     }

def qwen_vl_rollout(prompts, trainer):
    model = trainer.model
    processor = trainer.processing_class
    device = model.device

    dataset_batch = trainer._current_inputs

    prompt_ids_list = []
    completion_ids_list = []
    logprobs_list = []

    for i, prompt in enumerate(prompts):
        image = dataset_batch[i]["image_data"]

        for _ in range(trainer.num_generations):

            # === Build inputs ===
            messages = build_messages(image, prompt, system_prompt=SYSTEM_PROMPT)
            inputs, _ = build_inputs(processor, messages, device)

            # === Generate ===
            model.eval()
            with torch.no_grad():
                generated_ids = model.generate(
                    **inputs,
                    max_new_tokens=trainer.args.max_completion_length,
                    do_sample=True,
                    temperature=trainer.args.temperature,
                    top_p=trainer.args.top_p,
                )
            model.train()

            input_ids = inputs["input_ids"][0]  # (P)
            gen_ids = generated_ids[0, input_ids.shape[0]:]  # (G)

            # === Compute logprobs ===
            full_ids = torch.cat([input_ids, gen_ids], dim=0).unsqueeze(0)
            attention_mask = torch.ones_like(full_ids)

            with torch.no_grad():
              outputs = model(
                  input_ids=full_ids,
                  attention_mask=attention_mask,
              )

            logits = outputs.logits
            log_probs = torch.log_softmax(logits, dim=-1)

            prompt_len = input_ids.shape[0]
            token_logprobs = []

            for t in range(len(gen_ids)):
                token_id = int(gen_ids[t].item())
                pos = prompt_len + t - 1
                lp = log_probs[0, pos, token_id]
                token_logprobs.append(lp)

            token_logprobs = torch.stack(token_logprobs)

            # === Store tensors (NOT lists) ===
            prompt_ids_list.append(input_ids)
            completion_ids_list.append(gen_ids)
            logprobs_list.append(token_logprobs)

    # ===============================
    # 🔥 PAD EVERYTHING (CRITICAL)
    # ===============================

    pad_id = processor.tokenizer.pad_token_id or 0

    prompt_ids = torch.nn.utils.rnn.pad_sequence(
        prompt_ids_list, batch_first=True, padding_value=pad_id
    )

    completion_ids = torch.nn.utils.rnn.pad_sequence(
        completion_ids_list, batch_first=True, padding_value=pad_id
    )

    logprobs = torch.nn.utils.rnn.pad_sequence(
        logprobs_list, batch_first=True, padding_value=0.0
    )

    print("prompt ids ", prompt_ids.shape)
    print("completion ids ", completion_ids.shape)
    print("logprobs ", logprobs.shape)

    return {
        "prompt_ids": prompt_ids,
        "completion_ids": completion_ids,
        "logprobs": logprobs,
    }


class QwenVLGRPOTrainer(GRPOTrainer):

    def _prepare_inputs(self, inputs):
        # store raw batch for rollout access
        self._current_inputs = inputs

        cleaned_inputs = []

        for x in inputs:
            cleaned_inputs.append({
                "prompt": x["prompt"],
                "solution": x["solution"],
                # remove image_data
            })

        print("INPUT KEYS:", cleaned_inputs[0].keys())
        for k, v in cleaned_inputs[0].items():
          print(k, type(v))

        return super()._prepare_inputs(cleaned_inputs)

### Model

In [16]:
# model_name = "Qwen/Qwen2-VL-2B-Instruct"
# model_name = "Qwen/Qwen2.5-VL-3B-Instruct"
model_name = "Qwen/Qwen2.5-VL-7B-Instruct"
processor = AutoProcessor.from_pretrained(model_name)
# model = Qwen2VLForConditionalGeneration.from_pretrained(
#     model_name,
#     torch_dtype="auto",
#     device_map="auto"
# )

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("DEVICE = ", model.device)
model.train()




The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

DEVICE =  cuda:0


Qwen2_5_VLForConditionalGeneration(
  (model): Qwen2_5_VLModel(
    (visual): Qwen2_5_VisionTransformerPretrainedModel(
      (patch_embed): Qwen2_5_VisionPatchEmbed(
        (proj): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
      )
      (rotary_pos_emb): Qwen2_5_VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-31): 32 x Qwen2_5_VLVisionBlock(
          (norm1): Qwen2RMSNorm((1280,), eps=1e-06)
          (norm2): Qwen2RMSNorm((1280,), eps=1e-06)
          (attn): Qwen2_5_VLVisionAttention(
            (qkv): Linear(in_features=1280, out_features=3840, bias=True)
            (proj): Linear(in_features=1280, out_features=1280, bias=True)
          )
          (mlp): Qwen2_5_VLMLP(
            (gate_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (up_proj): Linear(in_features=1280, out_features=3420, bias=True)
            (down_proj): Linear(in_features=3420, out_features=1280, bias=True)
            (act_fn): SiLUAc

### Lora (optional)

In [17]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 5,046,272 || all params: 8,297,212,928 || trainable%: 0.0608


### Test basic model generation

In [18]:
# from mllm_generation import generate_pipeline
# from pprint import pprint

# temp_scene_graph = generate_pipeline(sample_orig, model, processor, sys_prompt=SYSTEM_PROMPT)
# pprint(temp_scene_graph)

In [19]:
# test tokenizer
print(processor.tokenizer.pad_token_id)

151643


### GRPO Config

In [20]:
# config = GRPOConfig(
#     output_dir="./grpo-sgg",

#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=2,

#     num_generations=2,   # keep small first
#     max_completion_length=1024,

#     learning_rate=1e-6,
#     beta=0.01,

#     temperature=0.8,
#     top_p=0.9,

#     logging_steps=1,
#     save_steps=50,

#     remove_unused_columns=False,
# )

def grpo_data_collator(features):
    batch = {}

    batch["prompt"] = [f["prompt"] for f in features]
    batch["image_data"] = [f["image_data"] for f in features]
    batch["solution"] = [f["solution"] for f in features]

    return batch

config = GRPOConfigVL(
    output_dir="./grpo-run",

    per_device_train_batch_size=1,   # B
    num_train_epochs=1,

    learning_rate=1e-6,

    num_generations=2,               # G
    max_completion_length=1024,

    logging_steps=1,
    save_steps=50,

    remove_unused_columns=False,
)

### Training

In [21]:
# trainer = QwenVLGRPOTrainer(
#     model=model,
#     reward_funcs=grpo_reward,
#     args=config,
#     train_dataset=train_ds,
#     processing_class=processor,
#     rollout_func=qwen_vl_rollout
# )

trainer = GRPOTrainerVL(
    model=model,
    args=config,
    train_dataset=train_ds,
    processing_class=processor,
    reward_funcs=grpo_reward,
    data_collator=grpo_data_collator,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 151645, 'bos_token_id': None, 'pad_token_id': 151643}.


GEN  0
GEN  1
GENERATED --- 
2  = prompt ids  <class 'torch.Tensor'>
2  = completion ids  <class 'torch.Tensor'>
2  = texts

--- SAMPLE 0 ---
TEXT: ```json
{
  "objects": [
    {"id": "whiteboard.1", "bbox": [285, 83, 492, 224]},
    {"id": "projector.2", "bbox": [60, 204, 98, 212]},
    {"id": "desk.3", "bbox": [100, 219, 702, 272]},
    {"id": "chair.4", "bbox": [262, 215, 314, 238]},
    {"id": "chair.5", "bbox": [213, 279, 345, 452]},
    {"id": "chair.6", "bbox": [333, 251, 464, 432]},
    {"id": "chair.7", "bbox": [429, 234, 485, 303]},
    {"id": "chair.8", "bbox": [505, 222, 557, 308]},
    {"id": "chair.9", "bbox": [549, 265, 625, 376]},
    {"id": "chair.10", "bbox": [624, 245, 692, 309]},
    {"id": "chair.11", "bbox": [632, 286, 804, 447]},
    {"id": "desk.12", "bbox": [134, 258, 811, 454]},
    {"id": "desk.13", "bbox": [444, 408, 811, 588]}
  ],
  "relationships": [
    {"subject": "whiteboard.1", "predicate": "mounted on", "object": "wall.1"},
    {"subject": "chair.4",

Step,Training Loss
1,-0.631462
2,-0.009524
3,2.606155
4,-6.150002
5,-0.889987
6,-0.450754
7,9.245453
8,-93.416664
9,1.049988
10,0.339392


GEN  0
GEN  1
GENERATED --- 
2  = prompt ids  <class 'torch.Tensor'>
2  = completion ids  <class 'torch.Tensor'>
2  = texts

--- SAMPLE 0 ---
TEXT: <answer>
{
  "objects": [
    {"id": "person.1", "bbox": [75, 69, 188, 248]},
    {"id": "person.2", "bbox": [138, 41, 252, 248]},
    {"id": "piano.1", "bbox": [0, 102, 187, 256]},
    {"id": "plant.1", "bbox": [225, 2, 279, 125]},
    {"id": "plant.2", "bbox": [11, 1, 102, 247]},
    {"id": "wall.1", "bbox": [0, 1, 279, 256]}
  ],
  "relationships": [
    {"subject": "person.1", "predicate": "playing", "object": "piano.1"},
    {"subject": "person.2", "predicate": "standing behind", "object": "person.1"},
    {"subject": "person.2", "predicate": "holding", "object": "person.1"},
    {"subject": "person.1", "predicate": "wearing", "object": "shirt.1"},
    {"subject": "person.2", "predicate": "wearing", "object": "dress.1"},
    {"subject": "piano.1", "predicate": "in front of", "object": "person.1"},
    {"subject": "plant.2", "predicate"

KeyboardInterrupt: 